# v16 — hierarki ECGFounder (sinus vs AF) lalu RR / fitur atrium (AFIB vs AFL)

Rancangan dikunci sebelum metrik dihitung: [RANCANGAN.md](RANCANGAN.md) (sha256 dan waktu di `../logs/v16_praregistrasi.log`).
Logika di `hierarki.py`. Semua masukan dibaca dari cache v3.3, v10, v12, v13, v15; tidak ada inferensi ECGFounder baru.
`../cache/v16/` hanya menyimpan OOF tahap 2 atrium dan prediksi paket v3.1 untuk Icentia.

| Sistem | Tahap 1 (sinus vs AF) | Tahap 2 (AFIB vs AFL) |
|---|---|---|
| **H-0 (primer)** | ECGFounder zero-shot, **tanpa penyetelan** (AF bila d1 > 0) | RR v3.1, b2 cross-fit |
| **H-A** | sama dengan H-0 | regresi logistik fitur atrium (v15), b2 cross-fit |
| **H-X** | skor log [pN, pAF·qAFIB, pAF·qAFL], bias 3 kelas cross-fit (tahap 1 ikut disetel di AFDB) | RR v3.1 |

In [1]:
import json, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import hierarki as H
from hierarki import cv, T, Ev, D, E13, K15, K33


def ci3(v):
    return f"{v[0]:.3f} [{v[1]:.2f}–{v[2]:.2f}]"


def dl(v):
    return f"Δ {v[0]:+.3f} [{v[1]:+.3f}…{v[2]:+.3f}], P = {v[3]:.3f} → {Ev.vonis(v[3])}"


d = D.muat()
R = len(d.folds)
lo_afdb, _ = E13.muat_afdb("50hz")
pE, d1 = H.tahap1(lo_afdb)
d2 = {"H-0": H.d2_rr(d.oof_v31), "H-A": H.oof_atrium()}
print("window AFDB:", len(d.y), "| ulangan:", R)

  atrium ulangan 0 selesai


  atrium ulangan 1 selesai


  atrium ulangan 2 selesai


  atrium ulangan 3 selesai


  atrium ulangan 4 selesai
window AFDB: 165978 | ulangan: 5


## 1. AFDB 3 kelas (H2, H3)

v13 zero-shot dan v15 gabungan dihitung ulang dari cache sebagai pembanding versi sebelumnya; angkanya di-assert sama dengan ringkasan versi itu.

In [2]:
pred, b2, bias = {}, {}, {}
pred["v1 (argmax)"] = d.oof_v1.argmax(-1)
pred["v3.1"], bias["v3.1"] = Ev.keputusan(d.oof_v31, d.folds, d.y)
oof_v13 = np.repeat(E13.skor_zero_shot(lo_afdb)[None], R, 0)
pred["v13 zero-shot"], bias["v13 zero-shot"] = Ev.keputusan(oof_v13, d.folds, d.y)
oof_v15 = K15.h2_afdb()
pred["v15 gabungan"], bias["v15 gabungan"] = Ev.keputusan(oof_v15, d.folds, d.y)
for k in ("H-0", "H-A"):
    pred[k], b2[k] = H.keputusan_h(d.y, d1, d2[k], d.folds)
oof_hx = H.skor_hx(pE, d.oof_v31)
pred["H-X"], bias["H-X"] = Ev.keputusan(oof_hx, d.folds, d.y)
tab = Ev.tabel_metrik(d, pred)
r13 = json.loads((E13.CACHE / "ringkasan.json").read_text())
r15 = json.loads((K15.CACHE / "ringkasan.json").read_text())
assert np.allclose(tab["v3.1"]["Macro-F1"], r13["metrik"]["v31"]["Macro-F1"])
assert np.allclose(tab["v13 zero-shot"]["Macro-F1"], r13["metrik"]["zero_50hz"]["Macro-F1"])
assert np.allclose(tab["v15 gabungan"]["Macro-F1"], r15["h2_afdb"]["metrik"]["gabungan"]["Macro-F1"])
tahap1 = dict(se=float((d1[d.y > 0] > 0).mean()), sp=float((d1[d.y == 0] <= 0).mean()),
              se_afib=float((d1[d.y == 1] > 0).mean()), se_afl=float((d1[d.y == 2] > 0).mean()))
print("tahap 1 tanpa penyetelan (AFDB): AF/AFL terdeteksi", round(tahap1["se"], 3), "(AFIB", round(tahap1["se_afib"], 3),
      ", AFL", round(tahap1["se_afl"], 3), ") | Normal benar", round(tahap1["sp"], 3))
pd.DataFrame({k: {m: ci3(v) for m, v in tab[k].items() if m in ("Macro-F1", "AFL F1", "AFL R", "AFL P", "AFIB F1", "Normal R")}
              for k in pred}).T

tahap 1 tanpa penyetelan (AFDB): AF/AFL terdeteksi 0.893 (AFIB 0.906 , AFL 0.153 ) | Normal benar 0.995


,Macro-F1,Normal R,AFIB F1,AFL P,AFL R,AFL F1
v1 (argmax),0.744 [0.64–0.81],0.903 [0.82–0.95],0.898 [0.85–0.93],0.275 [0.05–0.46],0.728 [0.37–0.98],0.399 [0.09–0.59]
v3.1,0.832 [0.67–0.92],0.924 [0.84–0.97],0.909 [0.86–0.94],0.715 [0.26–0.95],0.592 [0.13–0.91],0.647 [0.17–0.89]
v13 zero-shot,0.663 [0.64–0.68],0.973 [0.96–0.99],0.931 [0.85–0.97],0.053 [0.03–0.06],0.256 [0.03–0.36],0.087 [0.03–0.10]
v15 gabungan,0.756 [0.66–0.86],0.927 [0.84–0.97],0.915 [0.87–0.95],0.521 [0.17–0.72],0.346 [0.15–0.75],0.413 [0.18–0.71]
H-0,0.650 [0.62–0.67],0.995 [0.99–1.00],0.941 [0.91–0.97],0.065 [0.00–0.13],0.034 [0.00–0.05],0.044 [0.00–0.07]
H-A,0.649 [0.63–0.68],0.995 [0.99–1.00],0.940 [0.91–0.97],0.055 [0.01–0.14],0.037 [0.00–0.15],0.043 [0.01–0.12]
H-X,0.709 [0.63–0.75],0.925 [0.86–0.97],0.924 [0.86–0.96],0.191 [0.03–0.28],0.364 [0.11–0.51],0.250 [0.06–0.35]


In [3]:
bd = {k: Ev.banding(d, pred[k], pred["v3.1"]) for k in pred if k != "v3.1"}
bd["H-A vs H-0"] = Ev.banding(d, pred["H-A"], pred["H-0"])
display(pd.DataFrame({k: {m: dl(v) for m, v in b.items()} for k, b in bd.items()}).T.rename_axis("vs v3.1 (kecuali baris terakhir)"))
afl = {k: Ev.afl_per_pasien(d, p) for k, p in pred.items()}
palsu = {k: Ev.afl_palsu(d, p) for k, p in pred.items()}
display(pd.DataFrame(afl).round(2).rename_axis("recall AFL per pasien"))
display(pd.DataFrame(palsu).round(0).rename_axis("AFL palsu (window/ulangan)"))
info = {k: dict(median=float(np.median(v)), min=float(v.min()), maks=float(v.max()),
                di_batas=float(np.mean((v <= H.GRID[0]) | (v >= H.GRID[-1])))) for k, v in b2.items()}
info.update({k: dict(median=np.median(v.reshape(-1, 3), 0).tolist(), di_batas=Ev.bias_di_batas(v)) for k, v in bias.items()})
pd.DataFrame(info).T.rename_axis("b2 (H-0, H-A) / bias 3 kelas")

,Macro-F1,AFL F1,AFL P,AFL R,Normal R
vs v3.1 (kecuali baris terakhir),,,,,
v1 (argmax),"Δ -0.088 [-0.164…-0.018], P = 0.015 → lebih buruk","Δ -0.249 [-0.471…-0.049], P = 0.019 → lebih buruk","Δ -0.440 [-0.648…-0.219], P = 0.002 → lebih buruk","Δ +0.136 [+0.059…+0.245], P = 1.000 → lebih baik","Δ -0.020 [-0.033…-0.010], P = 0.000 → lebih buruk"
v13 zero-shot,"Δ -0.168 [-0.258…+0.005], P = 0.026 → lebih buruk","Δ -0.560 [-0.850…-0.086], P = 0.019 → lebih buruk","Δ -0.663 [-0.890…-0.217], P = 0.004 → lebih buruk","Δ -0.336 [-0.878…+0.182], P = 0.063 → tidak te...","Δ +0.049 [+0.001…+0.129], P = 0.980 → lebih baik"
v15 gabungan,"Δ -0.076 [-0.112…+0.014], P = 0.037 → lebih buruk","Δ -0.235 [-0.335…+0.032], P = 0.032 → lebih buruk","Δ -0.194 [-0.339…-0.082], P = 0.015 → lebih buruk","Δ -0.245 [-0.333…+0.092], P = 0.064 → tidak te...","Δ +0.003 [-0.003…+0.009], P = 0.852 → tidak te..."
H-0,"Δ -0.182 [-0.284…-0.015], P = 0.020 → lebih buruk","Δ -0.603 [-0.890…-0.162], P = 0.001 → lebih buruk","Δ -0.650 [-0.927…-0.252], P = 0.001 → lebih buruk","Δ -0.558 [-0.906…-0.091], P = 0.001 → lebih buruk","Δ +0.071 [+0.029…+0.150], P = 1.000 → lebih baik"
H-A,"Δ -0.183 [-0.283…+0.003], P = 0.026 → lebih buruk","Δ -0.604 [-0.883…-0.053], P = 0.019 → lebih buruk","Δ -0.660 [-0.904…-0.167], P = 0.009 → lebih buruk","Δ -0.555 [-0.906…-0.009], P = 0.025 → lebih buruk","Δ +0.071 [+0.029…+0.150], P = 1.000 → lebih baik"
H-X,"Δ -0.123 [-0.251…-0.020], P = 0.015 → lebih buruk","Δ -0.397 [-0.777…-0.083], P = 0.016 → lebih buruk","Δ -0.524 [-0.825…-0.227], P = 0.002 → lebih buruk","Δ -0.227 [-0.771…+0.050], P = 0.197 → tidak te...","Δ +0.001 [-0.082…+0.097], P = 0.515 → tidak te..."
H-A vs H-0,"Δ -0.001 [-0.012…+0.036], P = 0.461 → tidak te...","Δ -0.001 [-0.032…+0.107], P = 0.488 → tidak te...","Δ -0.010 [-0.077…+0.126], P = 0.414 → tidak te...","Δ +0.003 [-0.020…+0.128], P = 0.627 → tidak te...","Δ +0.000 [+0.000…+0.000], P = 0.000 → lebih buruk"


,v1 (argmax),v3.1,v13 zero-shot,v15 gabungan,H-0,H-A,H-X
recall AFL per pasien,,,,,,,
04043,1.00,0.40,0.80,0.20,0.00,0.00,1.00
04908,1.00,0.92,0.28,0.69,0.00,0.00,0.46
04936,0.67,0.51,0.35,0.16,0.05,0.03,0.52
06426,0.73,0.17,0.13,0.47,0.10,0.17,0.46
07910,0.21,0.00,0.40,0.10,0.00,0.18,0.10
08215,0.04,0.00,0.00,0.00,0.00,0.00,0.00
08378,1.00,0.96,0.01,0.78,0.00,0.00,0.10


,v1 (argmax),v3.1,v13 zero-shot,v15 gabungan,H-0,H-A,H-X
AFL palsu (window/ulangan),,,,,,,
total,2240.0,274.0,5320.0,307.0,583.0,766.0,1746.0
dari_normal,521.0,55.0,826.0,194.0,6.0,10.0,144.0
dari_afib,1719.0,219.0,4494.0,113.0,578.0,756.0,1602.0
pasien_tanpa_afl,1059.0,25.0,367.0,125.0,285.0,68.0,876.0


,median,min,maks,di_batas
"b2 (H-0, H-A) / bias 3 kelas",,,,
H-0,-1.25,-2.25,0.5,0.0
H-A,-1.0,-1.5,-0.75,0.0
v3.1,"[0.0, -0.75, -4.5]",NaN,NaN,0.0
v13 zero-shot,"[0.0, 1.75, 3.0]",NaN,NaN,0.42
v15 gabungan,"[0.0, -0.5, -2.25]",NaN,NaN,0.0
H-X,"[0.0, 3.0, 3.0]",NaN,NaN,0.82


## 2. AFDB 2 kelas (H2b), bootstrap bersarang dan tanpa 04936 (butir 10), metrik episode (butir 7)

In [4]:
dT = dict(y=d.y, rec=d.records, patients=d.patients, strata=d.strata, folds=d.folds)
W = cv.bootstrap_weights(d.strata, Ev.N_BOOT, seed=D.SEED)
W1 = np.vstack([np.ones(len(d.patients)), W])
pfold = np.array([[int(d.folds[r][d.records == p][0]) for p in d.patients] for r in range(R)])
cand_v31, _ = T.candidate_cms(dT, d.oof_v31)
dua = H.dua_kelas(d, d1, pred["v3.1"], W, W1, pfold, cand_v31)
display(pd.DataFrame({k: {m: ci3(v) for m, v in dua[k].items()} for k in ("H-0", "v3.1")}).T.rename_axis("2 kelas"))
display(pd.DataFrame({m: dl(v) for m, v in dua["banding"].items()}, index=["H-0 − v3.1"]).T)
print("bersarang 2 kelas:", {k: (np.round(v, 3).tolist() if isinstance(v, list) else round(v, 3)) for k, v in dua["bersarang"].items()})

,Macro-F1,F1 AF1,Se,Sp,PPV,F1 Normal
2 kelas,,,,,,
H-0,0.951 [0.92–0.98],0.940 [0.91–0.97],0.893 [0.83–0.95],0.995 [0.99–1.00],0.992 [0.99–1.00],0.963 [0.94–0.98]
v3.1,0.926 [0.88–0.95],0.913 [0.87–0.94],0.934 [0.91–0.96],0.924 [0.84–0.97],0.892 [0.81–0.95],0.938 [0.89–0.97]


,H-0 − v3.1
Macro-F1,"Δ +0.026 [-0.008…+0.070], P = 0.920 → tidak te..."
F1 AF1,"Δ +0.028 [-0.013…+0.077], P = 0.892 → tidak te..."
Se,"Δ -0.040 [-0.086…-0.002], P = 0.017 → lebih buruk"
Sp,"Δ +0.071 [+0.029…+0.150], P = 1.000 → lebih baik"


bersarang 2 kelas: {'titik': 0.951, 'ci': [0.924, 0.976], 'delta': 0.026, 'ci_delta': [-0.007, 0.086], 'p': 0.924}


In [5]:
nest = {"v3.1": H.bersarang3(cand_v31, pfold, W1)}
ref = T.nested_bootstrap(dT, cand_v31, {})
assert np.isclose(nest["v3.1"][0], ref["v31_point"]) and np.allclose(np.percentile(nest["v3.1"][1:], [2.5, 97.5]), ref["v31_ci"])
nest["H-0"] = H.bersarang3(H.kandidat_h(d.y, d1, d2["H-0"], d.records, d.patients), pfold, W1)
nest["H-X"] = H.bersarang3(T.candidate_cms(dT, oof_hx)[0], pfold, W1)
for k in nest:
    assert abs(nest[k][0] - tab[k]["Macro-F1"][0]) < 1e-9, k
bersarang = {k: H.banding_bersarang(nest[k], nest["v3.1"]) for k in ("H-0", "H-X")}
tanpa = H.tanpa_04936(d, d1, {"H-0": d2["H-0"]}, {"v31": d.oof_v31, "H-X": oof_hx})
ep = {k: H.episode(d, pred[k]) for k in ("v3.1", "H-0", "H-A", "H-X")}
display(pd.DataFrame(bersarang).T.rename_axis("bersarang vs v3.1"))
display(pd.DataFrame({k: {m: ci3(v) for m, v in r.items()} for k, r in tanpa["rows"].items()}).T.rename_axis("tanpa 04936"))
print("tanpa 04936, v3.1 − sistem [Δ, CI, P(v3.1 lebih baik)]:", {k: np.round(v, 3).tolist() for k, v in tanpa["pairs_v31_minus"].items()})
pd.DataFrame({f"{k} {c}": v for k, e in ep.items() for c, v in e.items()}).T.round(3)

,titik,ci,delta,ci_delta,p
bersarang vs v3.1,,,,,
H-0,0.649534,"[0.621178, 0.66387]",-0.182015,"[-0.273991, -0.012597]",0.0195
H-X,0.708595,"[0.628266, 0.767963]",-0.122954,"[-0.249643, -0.000432]",0.025


,Macro-F1,AFL F1,AFL P,AFL R,Normal R
tanpa 04936,,,,,
v31,0.859 [0.62–0.92],0.725 [0.02–0.90],0.805 [0.05–0.97],0.663 [0.02–0.89],0.928 [0.86–0.97]
H-X,0.651 [0.62–0.68],0.063 [0.02–0.11],0.040 [0.01–0.09],0.149 [0.10–0.44],0.948 [0.88–0.99]
H-0,0.637 [0.62–0.65],0.004 [0.00–0.02],0.004 [0.00–0.03],0.004 [0.00–0.03],0.995 [0.99–1.00]


tanpa 04936, v3.1 − sistem [Δ, CI, P(v3.1 lebih baik)]: {'H-X': [0.208, -0.014, 0.261, 0.946], 'H-0': [0.222, -0.018, 0.292, 0.93]}


,se_episode,ppv_episode,se_durasi,ppv_durasi,n_ref,n_det,palsu_per_jam
v3.1 AFIB,0.970,0.562,0.932,0.887,881.0,6252.4,11.858
v3.1 AFL,0.686,0.403,0.592,0.715,14.0,198.2,0.521
H-0 AFIB,0.917,0.939,0.897,0.990,881.0,4601.0,1.212
H-0 AFL,0.229,0.083,0.034,0.065,14.0,418.4,1.670
H-A AFIB,0.920,0.941,0.895,0.990,881.0,4737.0,1.218
H-A AFL,0.300,0.051,0.037,0.055,14.0,599.4,2.473
H-X AFIB,0.970,0.508,0.956,0.894,881.0,4536.4,9.683
H-X AFL,0.643,0.103,0.364,0.191,14.0,1037.0,4.045


## 3. Sinus takikardia asli (H1) dan keluaran keempat pada latihan fisik (H5)

Urutan window dicek dengan mengulang alarm zero-shot v13 persis. H-A punya tahap 1 yang sama dengan H-0, jadi alarmnya sama.

In [6]:
L = H.latihan()
J = L["J"]
m120 = L["hr"] >= 120
b_zero = E13.bias_median(oof_v13, d.y, d.folds)
pz = cv.apply_bias(E13.skor_zero_shot(L["logit"]), b_zero).argmax(1)
t, ci, _ = K33._boot_subjek(L["subjek"][m120], (pz[m120] != 0).astype(float))
assert np.isclose(t, r13["h2_latihan"]["zero-shot"]["alarm"]) and np.allclose(ci, r13["h2_latihan"]["zero-shot"]["ci"])
b2_med = {k: float(np.median(v)) for k, v in b2.items()}
bias_hx = E13.bias_median(oof_hx, d.y, d.folds)
kl, d1l = H.keputusan_eksternal(L["logit"], L["p31"], b2_med["H-0"], bias_hx)
h1 = {}
for k, v in kl.items():
    t, ci, _ = K33._boot_subjek(L["subjek"][m120], (v[m120] != 0).astype(float))
    h1[k] = dict(alarm=t, ci=ci.tolist(), n=int(m120.sum()),
                 per_sumber={s: float((v[m120 & (L["sumber"] == s)] != 0).mean()) for s in np.unique(L["sumber"][m120])},
                 pita=K33.alarm_per_pita(J, H.kelas_latihan(L, v), L["mask"]))
for k in ("H-0", "H-X"):
    h1[k]["banding"] = K33.banding_alarm(J, H.kelas_latihan(L, kl[k]), H.kelas_latihan(L, kl["v3.1 paket"]), L["mask"])
h1["H-0"]["lolos"] = bool(h1["H-0"]["alarm"] <= 0.5 * h1["v3.1 paket"]["alarm"] and h1["H-0"]["banding"]["ci"][1] < 0)
cek_l = (d1l <= 0) & (L["p31"].argmax(1) == 2)
t, ci, _ = K33._boot_subjek(L["subjek"][m120], cek_l[m120].astype(float))
h5_lat = dict(alarm=float((kl["H-0"][m120] != 0).mean()), cek=t, ci_cek=ci.tolist(),
              normal=float(((kl["H-0"][m120] == 0) & ~cek_l[m120]).mean()), n=int(m120.sum()))
display(pd.DataFrame({k: {"alarm ≥120 bpm": f"{v['alarm']*100:.1f}% [{v['ci'][0]*100:.0f}–{v['ci'][1]*100:.0f}]",
                          **{s: f"{x*100:.0f}%" for s, x in v["per_sumber"].items()}} for k, v in h1.items()}).T)
display(pd.DataFrame({k: {p: f"{x['alarm']*100:.0f}%" for p, x in v["pita"].items()} for k, v in h1.items()}).T.rename_axis("alarm per pita"))
print("H-0 − v3.1 (≥120 bpm):", {k: (np.round(x, 3).tolist() if isinstance(x, list) else round(x, 3)) for k, x in h1["H-0"]["banding"].items()},
      "| H1 lolos:", h1["H-0"]["lolos"])
print("keluaran keempat ≥120 bpm:", {k: (np.round(x, 3).tolist() if isinstance(x, list) else round(x, 3)) for k, x in h5_lat.items()})

,alarm ≥120 bpm,EPFL,EPHNOGRAM,Wrist
v3.1 paket,98.4% [97–100],100%,98%,100%
H-0,0.1% [0–0],0%,0%,0%
H-X,7.3% [4–11],6%,7%,3%


,0–100 bpm,100–110 bpm,110–120 bpm,120–130 bpm,130–140 bpm,140–150 bpm,150–∞ bpm
alarm per pita,,,,,,,
v3.1 paket,22%,16%,62%,97%,99%,99%,100%
H-0,1%,1%,0%,0%,0%,0%,0%
H-X,11%,9%,6%,5%,10%,7%,9%


H-0 − v3.1 (≥120 bpm): {'delta': -0.983, 'ci': [-0.995, -0.963], 'p_kurang': 1.0, 'n': 2189} | H1 lolos: True
keluaran keempat ≥120 bpm: {'alarm': 0.001, 'cek': 0.745, 'ci_cek': [0.611, 0.847], 'normal': 0.254, 'n': 2189}


## 4. Data eksternal (H3, H4)

Tahap 1 H-0/H-A tanpa parameter; `b2` dan bias H-X = median cross-fit AFDB; tahap 2 atrium dilatih di seluruh AFDB + SHDB-AF. Pembanding: paket v3.1.

In [7]:
P = H.ptbxl()
assert len(P["y"]) == r13["n"]["ptbxl"]
pzp = cv.apply_bias(E13.skor_zero_shot(P["logit"]), b_zero).argmax(1)
assert np.isclose(E13.banding_pasien(P["y"], pzp, P["p31"].argmax(1), P["pasien"])["titik_a"][0], r13["ptbxl"]["zero-shot"]["titik_a"][0])
model_a, med_a = H.model_atrium_penuh()
d2a = model_a.decision_function(K15.masukan(np.log(np.clip(P["p31"], 1e-7, 1)), P["fitur"], med_a))
kp, d1p = H.keputusan_eksternal(P["logit"], P["p31"], b2_med["H-0"], bias_hx, d2a, b2_med["H-A"])
ptb = {k: E13.banding_pasien(P["y"], kp[k], kp["v3.1 paket"], P["pasien"]) for k in ("H-0", "H-A", "H-X")}
ptb["H-A vs H-0"] = E13.banding_pasien(P["y"], kp["H-A"], kp["H-0"], P["pasien"])
ptb_st = {k: float((v[P["st"]] != 0).mean()) for k, v in kp.items()}
cek_p = (d1p <= 0) & (P["p31"].argmax(1) == 2)
h5_ptb = {n: dict(alarm=float((kp["H-0"][s] != 0).mean()), cek=float(cek_p[s].mean()), n=int(s.sum()))
          for n, s in (("STACH", P["st"]), ("AFL", P["y"] == 2), ("AFIB", P["y"] == 1))}
n_ptb = dict(window=int(len(P["y"])), afl=int((P["y"] == 2).sum()), pasien_afl=int(np.unique(P["pasien"][P["y"] == 2]).size),
             afib=int((P["y"] == 1).sum()), stach=int(P["st"].sum()))
display(pd.DataFrame({k: {"Macro-F1 (sistem / pembanding)": f"{v['titik_a'][0]:.3f} / {v['titik_b'][0]:.3f}",
                          "Δ Macro-F1, P": f"{v['Macro-F1'][0]:+.3f}, P = {v['Macro-F1'][3]:.3f} → {Ev.vonis(v['Macro-F1'][3])}",
                          "recall AFL (sistem / pembanding)": f"{v['titik_a'][1]:.2f} / {v['titik_b'][1]:.2f}",
                          "Δ recall AFL, P": f"{v['AFL R'][0]:+.2f}, P = {v['AFL R'][3]:.3f}"} for k, v in ptb.items()}).T.rename_axis("PTB-XL"))
print("n:", n_ptb)
print("alarm pada STACH:", {k: round(v, 3) for k, v in ptb_st.items()})
print("keluaran keempat PTB-XL:", h5_ptb)

,Macro-F1 (sistem / pembanding),"Δ Macro-F1, P",recall AFL (sistem / pembanding),"Δ recall AFL, P"
PTB-XL,,,,
H-0,0.687 / 0.622,"+0.065, P = 1.000 → lebih baik",0.14 / 0.50,"-0.36, P = 0.000"
H-A,0.703 / 0.622,"+0.080, P = 1.000 → lebih baik",0.18 / 0.50,"-0.32, P = 0.000"
H-X,0.578 / 0.622,"-0.045, P = 0.001 → lebih buruk",0.56 / 0.50,"+0.06, P = 0.728"
H-A vs H-0,0.703 / 0.687,"+0.015, P = 0.829 → tidak terbedakan",0.18 / 0.14,"+0.04, P = 0.769"


n: {'window': 20311, 'afl': 50, 'pasien_afl': 39, 'afib': 1443, 'stach': 811}
alarm pada STACH: {'v3.1 paket': 0.297, 'H-0': 0.016, 'H-X': 0.051, 'H-A': 0.016}
keluaran keempat PTB-XL: {'STACH': {'alarm': 0.016029593094944512, 'cek': 0.16646115906288533, 'n': 811}, 'AFL': {'alarm': 0.36, 'cek': 0.36, 'n': 50}, 'AFIB': {'alarm': 0.8801108801108801, 'cek': 0.004851004851004851, 'n': 1443}}


In [8]:
I = H.icentia()
ki, d1i = H.keputusan_eksternal(I["logit"], I["p31"], b2_med["H-0"], bias_hx)
assert np.isclose(E13.banding_pasien(I["y"], ki["v3.1 paket"], ki["v3.1 paket"], I["pasien"])["titik_a"][0], r13["icentia"]["zero-shot"]["titik_b"][0])
ico = {k: E13.banding_pasien(I["y"], ki[k], ki["v3.1 paket"], I["pasien"]) for k in ("H-0", "H-X")}
n_ico = dict(window=int(len(I["y"])), afl=int((I["y"] == 2).sum()), afib=int((I["y"] == 1).sum()), pasien=int(np.unique(I["pasien"]).size))
print("n:", n_ico)
pd.DataFrame({k: {"Macro-F1 (sistem / paket v3.1)": f"{v['titik_a'][0]:.3f} / {v['titik_b'][0]:.3f}",
                  "Δ Macro-F1, P": f"{v['Macro-F1'][0]:+.3f}, P = {v['Macro-F1'][3]:.3f} → {Ev.vonis(v['Macro-F1'][3])}",
                  "recall AFL (sistem / paket v3.1)": f"{v['titik_a'][1]:.2f} / {v['titik_b'][1]:.2f}"} for k, v in ico.items()}).T.rename_axis("Icentia11k")

n: {'window': 115722, 'afl': 53253, 'afib': 27733, 'pasien': 2420}


,Macro-F1 (sistem / paket v3.1),"Δ Macro-F1, P",recall AFL (sistem / paket v3.1)
Icentia11k,,,
H-0,0.449 / 0.476,"-0.027, P = 0.000 → lebih buruk",0.02 / 0.05
H-X,0.488 / 0.476,"+0.012, P = 0.989 → lebih baik",0.08 / 0.05


## 5. Keluaran keempat di AFDB (H5) dan AFL 2:1 vs sinus takikardia (H6)

In [9]:
h5 = H.h5_afdb(d, d1, pred["H-0"], pred["v3.1"])
h5.update({"latihan_≥120": h5_lat, "ptbxl": h5_ptb})
display(pd.DataFrame({**h5["kelas"], **{f"AFL {p}": v for p, v in h5["pasien_afl"].items()}}).T.round(3)
        .rename_axis("H-0 + 'perlu dicek' (AFDB)"))
h6 = H.h6(lo_afdb, d, L)
print("H6 AUROC AFL 08378 vs sinus ≥120 bpm:", {k: (np.round(v, 3).tolist() if isinstance(v, list) else (round(v, 3) if isinstance(v, float) else v)) for k, v in h6.items()})

,alarm,cek,normal,n
H-0 + 'perlu dicek' (AFDB),,,,
Normal,0.005,0.001,0.995,99000.0
AFIB,0.906,0.002,0.092,65826.0
AFL,0.153,0.577,0.270,1152.0
AFL 04043,0.000,0.400,0.600,1.0
AFL 04908,0.000,0.922,0.078,46.0
AFL 04936,0.184,0.484,0.332,673.0
AFL 06426,0.381,0.171,0.448,21.0
AFL 07910,0.458,0.000,0.542,96.0
AFL 08215,0.000,0.000,1.000,9.0


H6 AUROC AFL 08378 vs sinus ≥120 bpm: {'ecgfounder': 0.782, 'v31': 0.901, 'n_pos': 306, 'n_neg': 2189, 'median_ecgfounder': [-3.912, -5.289]}


In [10]:
kriteria = dict(H1=h1["H-0"]["lolos"], H2=bool(bd["H-0"]["Macro-F1"][3] > 0.05),
                H4_ptbxl=bool(ptb["H-0"]["Macro-F1"][3] > 0.05), H4_icentia=bool(ico["H-0"]["Macro-F1"][3] > 0.05))
kriteria["kandidat_uji_perangkat"] = all(kriteria.values())
ring = {"metrik": tab, "banding_v31": bd, "afl_per_pasien": afl, "afl_palsu": palsu, "parameter": info, "tahap1_afdb": tahap1,
        "dua_kelas": dua, "bersarang": bersarang, "tanpa_04936": tanpa, "episode": ep, "h1_latihan": h1, "ptbxl": ptb,
        "ptbxl_st": ptb_st, "n_ptbxl": n_ptb, "icentia": ico, "n_icentia": n_ico, "h5": h5, "h6": h6, "kriteria": kriteria,
        "b2_median": b2_med, "bias_hx_median": bias_hx.tolist(), "lingkungan": Ev._lingkungan()}
H.CACHE.mkdir(parents=True, exist_ok=True)
(H.CACHE / "ringkasan.json").write_text(json.dumps(ring, indent=1, default=float, ensure_ascii=False))
print("kriteria:", kriteria)
print("tersimpan:", H.CACHE / "ringkasan.json")

kriteria: {'H1': True, 'H2': False, 'H4_ptbxl': True, 'H4_icentia': False, 'kandidat_uji_perangkat': False}
tersimpan: /mnt/HDD/03_Akademik_dan_Tugas/WrapSihedaf/cache/v16/ringkasan.json


## 6. Kesimpulan

Dijalankan penuh 6 November 2026 di PC (RTX 5060, ±12 menit, semua masukan dari cache). Kriteria yang dikunci (RANCANGAN bagian 5):
H1 lolos, **H2 gagal**, H4 lolos di PTB-XL tetapi **gagal di Icentia** → v16 **bukan kandidat** sistem 3 kelas untuk uji perangkat.

- **Tahap 1 ECGFounder (tanpa penyetelan) sangat spesifik untuk AFIB, tetapi menyebut AFL sebagai sinus.** Di AFDB: AFIB terdeteksi
  90,6%, Normal benar 99,5%, AFL hanya 15,3%. Macro-F1 3 kelas H-0 0,650 vs v3.1 0,832 (lebih buruk, P = 0,020; bersarang P = 0,0195),
  recall AFL 0,03 (08378: 0,00). Tanpa 04936: 0,637 vs 0,859.
- **H1 lolos sangat jelas:** alarm AFIB/AFL pada sinus takikardia asli ≥ 120 bpm 0,1% [0–0] vs 98,4% (paket v3.1), Δ −0,98
  [−0,995…−0,963]. Alarm < 100 bpm saat bergerak (jitter detektor 50 Hz) juga turun 22% → 1%. PTB-XL STACH 29,7% → 1,6%.
- **2 kelas (Normal vs AF/AFL):** H-0 0,951 vs v3.1 0,926 (Δ +0,026, P = 0,92, tidak terbedakan; bersarang P = 0,92). Spesifisitas
  0,995 vs 0,924 (lebih baik), sensitivitas 0,893 vs 0,934 (lebih buruk, P = 0,017). Episode AFIB palsu 11,9 → 1,2 per jam.
- **H3 gagal:** tahap 2 atrium tidak berbeda dari RR (AFDB Δ −0,001, P = 0,46; PTB-XL Δ +0,015, P = 0,83), karena tahap 1 sudah
  membuang sebagian besar AFL sebelum tahap 2.
- **H4:** PTB-XL H-0 0,687 vs 0,622 (lebih baik, P = 1,00) tetapi recall AFL 0,14 vs 0,50; Icentia 0,449 vs 0,476 (lebih buruk).
  H-X (tahap 1 disetel di AFDB): bias di batas grid 82% (tanda bahaya), AFDB 0,709 (lebih buruk), alarm sinus takikardia 7,3%.
- **H5 (keluaran keempat "takikardia teratur, perlu dicek"):** menampung 96% window AFL 08378 dan 58% seluruh AFL AFDB, hanya 0,1%
  Normal AFDB, tetapi juga 74% [61–85] sinus takikardia asli ≥ 120 bpm dan 17% STACH PTB-XL. AFL 2:1 dan sinus takikardia tetap
  tidak terpisah; yang berubah hanya alarm AF menjadi pemberitahuan lunak.
- **H6:** logit ECGFounder AFL − ST memisahkan window AFL 08378 dari sinus takikardia asli dengan AUROC 0,78; skor RR v3.1 0,90 (beda
  dataset, jadi bisa tercampur pergeseran domain).

Implikasi: ECGFounder di backend layak dipertimbangkan sebagai detektor **AF 2 kelas yang sangat spesifik** bila produk menerima AFL
dilaporkan lewat jalur "perlu dicek". Itu keputusan produk/klinis, bukan hasil v16. Untuk 3 kelas, AFL pada EKG 50 Hz satu lead tetap
belum terpecahkan dengan data publik.